In [ ]:
# 1. Check your Colab CUDA version
!nvidia-smi | grep "CUDA Version"

# 2. Clean install matching torch + torchvision + torchaudio
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124   # change to cu121 or cu118 if your nvidia-smi says so

In [ ]:
!git clone https://github.com/MoonshotAI/Kimi-Audio.git
%cd Kimi-Audio
!git submodule update --init --recursive
!pip install -r requirements.txt
# FIX: Install a version known to work with recent GLM/Whisper logic but before v5 breaking changes
!pip install "transformers==4.42.4" "accelerate>=0.27.0"
!pip install -e .

In [ ]:
import sys
import transformers
from IPython.display import display, HTML

print(f"Current Transformers Version: {transformers.__version__}")

# Check if we are running the correct version
if transformers.__version__.startswith("5."):
    display(HTML("""
    <div style="background-color: #ffcccc; padding: 10px; border: 1px solid red; border-radius: 5px;">
        <h3>⚠️ ACTION REQUIRED: Restart Runtime</h3>
        <p>You are still running <b>Transformers v5.0.0</b>, which is incompatible.</p>
        <p>Please go to <b>Runtime > Restart session</b> to load the installed v4.42.4.</p>
    </div>
    """))
    raise RuntimeError("Please restart the runtime to load the correct transformers version.")

import torch
import warnings
from transformers import AutoModelForCausalLM, AutoConfig
import transformers.cache_utils

# --- Patching Missing EncoderDecoderCache ---
# Kimi-Audio expects EncoderDecoderCache, which might be missing in this transformers version.
if not hasattr(transformers.cache_utils, "EncoderDecoderCache"):
    print("🔧 Patching transformers.cache_utils.EncoderDecoderCache...")

    # Define a minimal mock or compatible class
    # EncoderDecoderCache usually manages self and cross attention caches
    class EncoderDecoderCache(torch.nn.Module):
        def __init__(self, self_attention_cache, cross_attention_cache):
            super().__init__()
            self.self_attention_cache = self_attention_cache
            self.cross_attention_cache = cross_attention_cache

        def get_seq_length(self, layer_idx=0):
            return self.self_attention_cache.get_seq_length(layer_idx)

        def to(self, device):
            self.self_attention_cache.to(device)
            self.cross_attention_cache.to(device)
            return self

    # Inject into the module
    transformers.cache_utils.EncoderDecoderCache = EncoderDecoderCache
    # Also inject into the module dictionary so 'from ... import ...' works
    sys.modules["transformers.cache_utils"].EncoderDecoderCache = EncoderDecoderCache

# --- Patching apply_rotary_pos_emb for Transformers v5+ compatibility ---
# PROBLEM: Newer Transformers changed the signature of apply_rotary_pos_emb.
# The model calls it as (q, k, cos, sin, position_ids), but the installed
# version expects (q, k, cos, sin, unsqueeze_dim=1).
# FIX: Wrap the function to accept (and silently drop) the extra positional arg.
try:
    from transformers.models.qwen2 import modeling_qwen2
    _original_apply = modeling_qwen2.apply_rotary_pos_emb

    def patched_apply_rotary_pos_emb(q, k, cos, sin, *args, **kwargs):
        # Ignore extra positional arg (position_ids from older calling convention)
        return _original_apply(q, k, cos, sin, unsqueeze_dim=1)

    modeling_qwen2.apply_rotary_pos_emb = patched_apply_rotary_pos_emb
    patched_count = 0
    for name, module in list(sys.modules.items()):
        if 'modeling_moonshot_kimia' in name and hasattr(module, 'apply_rotary_pos_emb'):
            module.apply_rotary_pos_emb = patched_apply_rotary_pos_emb
            patched_count += 1
    print(f"\u2705 apply_rotary_pos_emb patch applied (+ {patched_count} kimia module(s)).")
except Exception as e:
    print(f"\u26a0\ufe0f Could not apply apply_rotary_pos_emb patch: {e}")

# Import the KimiAudio class
try:
    from kimia_infer.api.kimia import KimiAudio
except ImportError:
    sys.path.append("/content/Kimi-Audio")
    from kimia_infer.api.kimia import KimiAudio

# --- Load Model ---
print("🚀 Initializing Kimi-Audio...")
try:
    model = KimiAudio("moonshotai/Kimi-Audio-7B-Instruct")
    print("\n✅ SUCCESS: Model loaded successfully! You can now run the inference cells.")
except Exception as e:
    print(f"\n❌ FATAL: Model load failed: {e}")
    raise e

In [ ]:
# ⚠️ EXECUTE THIS CELL TO RESTART THE RUNTIME ⚠️
import os
import time

print("♻️ Restarting runtime to load the correct libraries...")
print("1. The session will crash/restart (this is intentional).")
print("2. After it reconnects, SKIP the installation cells.")
print("3. Run the 'Load Model' cell directly.")

time.sleep(2)
os.kill(os.getpid(), 9)

In [ ]:
import soundfile as sf
import torch
import numpy as np
from IPython.display import Audio, display
import os
import traceback

# Helper function to run inference
def generate_response(audio_path, text_prompt="Respond naturally to the audio.", output_type="both"):
    """
    output_type: "text", "audio", or "both"
    """
    # Check if model is loaded
    if 'model' not in globals():
        print("Error: 'model' is not defined. Please run the cell that loads the KimiAudio model (KimiAudio(...)) before running inference.")
        return None, None

    messages = [
        {"role": "user", "message_type": "text", "content": text_prompt},
        {"role": "user", "message_type": "audio", "content": audio_path},
    ]

    print(f"Processing: {text_prompt} + {os.path.basename(audio_path)}")

    # API workaround: The model only accepts 'text' or 'both'
    # If user wants 'audio', we ask for 'both' and ignore text.
    effective_output_type = output_type
    if output_type == "audio":
        effective_output_type = "both"

    # Generate
    try:
        wav_output, text_output = model.generate(
            messages,
            output_type=effective_output_type,
            audio_temperature=0.8,
            audio_top_k=10,
            text_temperature=0.0,
            text_top_k=5,
            audio_repetition_penalty=1.0,
            audio_repetition_window_size=64,
            text_repetition_penalty=1.0,
            text_repetition_window_size=16,
        )
    except Exception:
        print("Error during generation:")
        traceback.print_exc()
        return None, None

    # Filter output if needed
    if output_type == "audio":
        text_output = None

    return wav_output, text_output

# Helper to save and display audio
def save_and_display(wav_output, text_output, filename="output.wav"):
    if wav_output is not None:
        # Detach and convert to numpy if it's a tensor
        if isinstance(wav_output, torch.Tensor):
            audio_data = wav_output.detach().cpu().view(-1).numpy()
        else:
            audio_data = wav_output

        # Save to file (24kHz sample rate is standard for this model)
        sf.write(filename, audio_data, 24000)
        print(f"Audio saved to: {filename}")

        # Display in Colab
        display(Audio(filename, rate=24000))

    if text_output:
        print(f"\n📝 Text Response:\n{text_output}")

    return filename

In [ ]:
import os
import sys

print("--- Checking Kimi-Audio Installation ---")

# 1. Check Directory
repo_path = "/content/Kimi-Audio"
if os.path.exists(repo_path):
    print(f"✅ Repo directory found at: {repo_path}")
else:
    print(f"❌ Repo directory NOT found at: {repo_path}")

# 2. Check Import
try:
    import kimia_infer
    print(f"✅ 'kimia_infer' module imported successfully.")
    print(f"   Location: {os.path.dirname(kimia_infer.__file__)}")
except ImportError as e:
    print(f"❌ Failed to import 'kimia_infer': {e}")
    # Try adding to path if missing (standard fallback)
    if repo_path not in sys.path:
        print("   (Adding repo path to sys.path and retrying...)")
        sys.path.append(repo_path)
        try:
            import kimia_infer
            print(f"✅ 'kimia_infer' module imported successfully after sys.path update.")
        except ImportError as e2:
             print(f"❌ Still failed to import: {e2}")

# 3. Check KimiAudio Class
try:
    from kimia_infer.api.kimia import KimiAudio
    print(f"✅ 'KimiAudio' class found.")
except ImportError as e:
    print(f"❌ Failed to import 'KimiAudio' class: {e}")

print("--- Check Complete ---")

In [ ]:
import sys
import transformers
from transformers.models.qwen2 import modeling_qwen2

# NOTE: This patch is now applied automatically in the 'Load Model' cell above.
# Run this cell only if you need to re-apply the patch to an already-loaded model.
print("--- Re-applying Runtime Patch for Transformers v5+ compatibility ---")

# PROBLEM: The installed 'apply_rotary_pos_emb' signature is (q, k, cos, sin, unsqueeze_dim=1)
# BUT: The model code calls it as (q, k, cos, sin, position_ids)
# FIX: We define a wrapper to ignore the 5th argument if it's passed positionally (position_ids)

_original_apply = modeling_qwen2.apply_rotary_pos_emb

def patched_apply_rotary_pos_emb(q, k, cos, sin, *args, **kwargs):
    # We ignore the extra positional arg (position_ids) and rely on the default unsqueeze_dim=1
    return _original_apply(q, k, cos, sin, unsqueeze_dim=1)

# 1. Patch the source library so future imports work
modeling_qwen2.apply_rotary_pos_emb = patched_apply_rotary_pos_emb

# 2. Patch the already loaded remote code module
patched_count = 0
for name, module in list(sys.modules.items()):
    if 'modeling_moonshot_kimia' in name:
        if hasattr(module, 'apply_rotary_pos_emb'):
            print(f"🔧 Patching loaded module: {name}")
            module.apply_rotary_pos_emb = patched_apply_rotary_pos_emb
            patched_count += 1

if patched_count > 0:
    print(f"✅ Success: Patched {patched_count} instance(s) of the model code.")
else:
    print("⚠️ Warning: Could not find loaded model module to patch. If the model isn't loaded yet, the library patch (#1) will handle it.")

In [ ]:
import numpy as np
import soundfile as sf
import os

# Create a dummy silence file (0.5s) to use as context
# This helps the model generate audio from text prompts without an actual input voice
silence_path = "/content/silence_0.5s.wav"
if not os.path.exists(silence_path):
    sr = 24000
    silence = np.zeros(int(0.5 * sr))
    sf.write(silence_path, silence, sr)
    print(f"Created dummy silence file: {silence_path}")

In [ ]:
import traceback

def generate_audio_event(prompt, output_filename, use_silence_context=True):
    """
    Generic function to generate audio from a text description.
    """
    # Safety check
    if 'model' not in globals():
        print(f"Skipping generation for '{output_filename}': Model not loaded.")
        print("Please run the cell 'lG1KcW28uiZh' to load the model first.")
        return

    messages = [
        {"role": "user", "message_type": "text", "content": prompt}
    ]

    # Add silent audio context if requested (often helps stability)
    if use_silence_context:
        messages.append(
            {"role": "user", "message_type": "audio", "content": "/content/silence_0.5s.wav"}
        )

    print(f"Generating: '{prompt}'...")

    try:
        # Generate audio only
        # FIX: The API throws an error if output_type="audio". We must use "both".
        wav_output, _ = model.generate(
            messages,
            output_type="both",
            audio_temperature=0.7,  # Slightly lower temp for cleaner instruments
            audio_top_k=10
        )

        # Save and display using the existing helper
        if wav_output is not None:
            save_and_display(wav_output, None, output_filename)
        else:
            print("Model returned no audio.")

    except Exception:
        print("Generation failed:")
        traceback.print_exc()

# --- 1. Instrument One-Shot Wrapper ---
def generate_instrument(instrument_name, description="clean single hit"):
    prompt = f"Sound of a {instrument_name}. {description}. High quality, isolated sample."
    filename = f"inst_{instrument_name.replace(' ', '_')}.wav"
    generate_audio_event(prompt, filename)

# --- 2. Singing Wrapper ---
def generate_singing(lyrics, style="pop"):
    prompt = f"Sing the following lyrics in a {style} style: \"{lyrics}\""
    filename = f"vocal_{style.split()[0]}.wav"
    generate_audio_event(prompt, filename)

In [ ]:
import torch
import numpy as np

print("--- Diagnostic: Token ID & RoPE Check (Final) ---")

if 'model' in globals():
    try:
        config = model.alm.config
        pm = model.prompt_manager

        print(f"\n1. Model Config:")
        print(f"   Vocab Size: {config.vocab_size}")
        print(f"   RoPE Theta: {getattr(config, 'rope_theta', 'MISSING')}")

        print("\n2. Generating prompt inputs...")
        test_messages = [{"role": "user", "message_type": "text", "content": "Testing inputs."}]
        history = pm.get_prompt(test_messages, output_type="both")

        # Check Text Token IDs
        if hasattr(history, 'text_token_ids'):
            ids = history.text_token_ids
            print(f"\n3. Checking Text Token IDs:")

            # Handle list vs tensor
            if isinstance(ids, list):
                print(f"   Type: List (Length: {len(ids)})")
                if len(ids) > 0:
                    max_id = max(ids)
                    print(f"   Max ID: {max_id}")

                    if max_id >= config.vocab_size:
                        print(f"\n❌ CRITICAL: Token ID {max_id} >= Vocab Size {config.vocab_size}!")
                        print("   Action: We must resize the model embeddings or fix the tokenizer.")
                    else:
                        print("\n✅ Token IDs are safe. The crash is likely due to RoPE/NaNs.")
                else:
                    print("   (Empty list)")
            elif isinstance(ids, torch.Tensor):
                print(f"   Type: Tensor (Shape: {ids.shape})")
                max_id = ids.max().item()
                print(f"   Max ID: {max_id}")
                if max_id >= config.vocab_size:
                     print(f"\n❌ CRITICAL: Token ID {max_id} >= Vocab Size {config.vocab_size}!")
                else:
                     print("\n✅ Token IDs are safe.")

    except Exception as e:
        print(f"\nDiagnostic failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ Model not loaded.")

In [ ]:
import os
import torch
import torchaudio
import traceback

# Ensure CUDA launch blocking for better error messages if we run model code later
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

def debug_audio_tokenization(audio_path):
    """Test if the audio file is valid before sending to model"""
    print(f"--- Inspecting Audio: {audio_path} ---")

    # 1. Check file exists and has content
    if not os.path.exists(audio_path):
        print("❌ File does not exist")
        return False

    size = os.path.getsize(audio_path)
    print(f"   File size: {size} bytes")
    if size < 100:
        print("❌ File too small (likely corrupted or empty header)")
        return False

    # 2. Load and inspect
    try:
        waveform, sample_rate = torchaudio.load(audio_path)
        print(f"✅ Loaded successfully.")
        print(f"   Shape: {waveform.shape} (Channels, Samples)")
        print(f"   Sample Rate: {sample_rate} Hz")
        print(f"   Dtype: {waveform.dtype}")

        # 3. Check for NaN/Inf
        if torch.isnan(waveform).any():
            print("❌ Waveform contains NaN values!")
            return False
        if torch.isinf(waveform).any():
            print("❌ Waveform contains Inf values!")
            return False

        # 4. Check statistics
        print(f"   Min: {waveform.min().item():.4f}, Max: {waveform.max().item():.4f}, Mean: {waveform.mean().item():.4f}")

        # 5. Check dimensions (should be [channels, samples])
        if waveform.dim() != 2:
            print(f"❌ Wrong dimensions: {waveform.dim()} (expected 2)")
            return False

        return True

    except Exception as e:
        print(f"❌ Failed to load audio with torchaudio: {e}")
        traceback.print_exc()
        return False

# Run the check on our context file
silence_file = "/content/silence_0.5s.wav"
debug_audio_tokenization(silence_file)

In [ ]:
# === Advanced Singing Examples ===
# Run this cell after the model is loaded to hear different styles

styles = [
    ("90s Rock", "I'm driving down the highway, looking for a place to go."),
    ("Folk acoustic", "The leaves are falling down, waiting for the winter snow."),
    ("Synth-pop", "Neon lights are flashing, dancing in the midnight rain.")
]

if 'generate_singing' in globals():
    print("--- Generating Varied Singing Styles ---")
    for style, lyrics in styles:
        print(f"\n🎤 Style: {style}")
        generate_singing(lyrics, style)
else:
    print("Please run the cell defining 'generate_singing' first (Cell e905d36b).")

In [ ]:
# Define paths to sample audio files included in the repo
asr_audio = "/content/Kimi-Audio/test_audios/asr_example.wav"
qa_audio = "/content/Kimi-Audio/test_audios/qa_example.wav"

# ==================== EXAMPLE 1: ASR (Transcription) ====================
if os.path.exists(asr_audio):
    print("=== Example 1: Speech Recognition ===")
    wav_out, txt_out = generate_response(
        audio_path=asr_audio,
        text_prompt="Transcribe the speech exactly as spoken.",
        output_type="text"
    )
    print(f"Transcription Result: {txt_out}")
else:
    print(f"File not found: {asr_audio}")

print("\n" + "="*40 + "\n")

# ==================== EXAMPLE 2: Audio Q&A ====================
if os.path.exists(qa_audio):
    print("=== Example 2: Audio Question Answering ===")
    wav_out, txt_out = generate_response(
        audio_path=qa_audio,
        text_prompt="Answer the question in the audio clearly.",
        output_type="both"
    )
    save_and_display(wav_out, txt_out, "qa_response.wav")
else:
    print(f"File not found: {qa_audio}")

In [ ]:
# ==================== QUICK TEST: Generated Audio ====================
import numpy as np
import soundfile as sf

# Create a simple test audio file (1 second beep)
sample_rate = 24000
t = np.linspace(0, 1, sample_rate)
test_audio = 0.3 * np.sin(2 * np.pi * 440 * t)  # 440Hz tone

output_path = "/content/test_beep.wav"
sf.write(output_path, test_audio, sample_rate)
print(f"Created test audio at {output_path}")

# Test ASR on the tone (model will likely say it's a tone/beep)
if 'generate_response' in locals():
    print("Running inference on test beep...")
    wav_out, txt_out = generate_response(
        output_path,
        "What sound is this?",
        output_type="both"
    )
    save_and_display(wav_out, txt_out, "beep_response.wav")
else:
    print("Error: 'generate_response' function not defined. Please run the helper function cell first.")